# 09 — kvpress: Qasper Benchmark with PrefillDecodingPress

This notebook evaluates KV cache compression on the
[Qasper](https://huggingface.co/datasets/tau/scrolls) benchmark (from SCROLLS)
using [kvpress](https://github.com/NVIDIA/kvpress) with Qwen3-8B.

Qasper tests document QA on full NLP research papers (~3K–8K tokens),
requiring both long-context comprehension (prefill) and answer generation
(decoding).

We test **KeyDiffPress**-based compression applied to both **prefill** and
**decoding** phases using PrefillDecodingPress, with two decoding strategies:
- **full_replacement** — CompressionRatioDecodingPress
- **filtering** — FilteringPress

Scoring uses the HuggingFace `evaluate` library (SQuAD F1).

Results are saved to `results/kvpress_qasper/` for comparison in later notebooks.

## Configuration

In [ ]:
import os
os.environ["HF_HOME"] = "/opt/app-root/src/.cache/huggingface"

MODEL_NAME = "Qwen/Qwen3-8B"

COMPRESSION_RATIOS = [0.01, 0.25, 0.50, 0.75]

FRACTION = 0.01

MAX_NEW_TOKENS = 64

PRESS_CONFIGS = {
    "full_replacement": lambda cr: PrefillDecodingPress(
        prefilling_press=KeyDiffPress(compression_ratio=cr),
        decoding_press=CompressionRatioDecodingPress(
            base_press=KeyDiffPress(), target_compression_ratio=cr,
        ),
    ),
    "filtering": lambda cr: PrefillDecodingPress(
        prefilling_press=KeyDiffPress(compression_ratio=cr),
        decoding_press=FilteringPress(
            base_press=KeyDiffPress(), target_compression_ratio=cr,
            fill_padding=False,
        ),
    ),
}

In [ ]:
import sys
import builtins

_original_print = builtins.print

def print(*args, **kwargs):
    _original_print(*args, **kwargs)
    if sys.stdout is not sys.__stdout__:
        kwargs['file'] = sys.__stdout__
        kwargs['flush'] = True
        _original_print(*args, **kwargs)

In [ ]:
import sys
import os

FORK_DIR = "/opt/app-root/src/kvpress-fork"

if os.path.isdir(FORK_DIR) and os.listdir(FORK_DIR):
    sys.path.insert(0, FORK_DIR)
    import kvpress
    print(f"Using FORK kvpress from {FORK_DIR}")
else:
    import kvpress
    print(f"Using SYSTEM kvpress")

print(f"  location: {os.path.dirname(kvpress.__file__)}")

## 1. Load Model

In [ ]:
import torch
from transformers import pipeline
from kvpress import (
    KeyDiffPress, PrefillDecodingPress, CompressionRatioDecodingPress,
    FilteringPress,
)

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {torch.cuda.get_device_name(0)} — {vram_gb:.1f} GB VRAM")

model_kwargs = {}

try:
    import flash_attn  # noqa: F401
    model_kwargs["attn_implementation"] = "flash_attention_2"
    print("Using Flash Attention 2")
except ImportError:
    print("Flash Attention 2 not available, using default attention")

pipe = pipeline(
    "kv-press-text-generation",
    model=MODEL_NAME,
    device_map="auto",
    model_kwargs=model_kwargs,
    trust_remote_code=True,
)

print(f"\nModel loaded. GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 2. Load Qasper Dataset

In [ ]:
from datasets import load_dataset

qasper_ds = load_dataset("tau/scrolls", "qasper", split="validation")
if FRACTION < 1.0:
    n = max(1, int(len(qasper_ds) * FRACTION))
    qasper_ds = qasper_ds.select(range(n))

print(f"Qasper dataset: {len(qasper_ds)} examples")
print(f"Columns: {qasper_ds.column_names}")
print(f"\nSample input (first 200 chars): {qasper_ds[0]['input'][:200]}...")
print(f"Sample output: {qasper_ds[0]['output']}")

## 3. Load Scoring Metric

In [ ]:
import evaluate

squad_metric = evaluate.load("squad")
print("Loaded SQuAD metric (token-level F1)")

## 4. Run Inference

For each (algorithm, compression_ratio) combination, run all Qasper
examples through the kvpress pipeline. The SCROLLS `input` field
already contains the full paper text with the question appended.

In [ ]:
import time

all_results = []

configs = [("no_press", 0.0, None)]
for press_name, press_factory in PRESS_CONFIGS.items():
    for ratio in COMPRESSION_RATIOS:
        configs.append((press_name, ratio, press_factory(ratio)))

for press_name, ratio, press in configs:
    label = f"{press_name} | ratio={ratio}"
    print(f"\n{'='*60}")
    print(f"Running: {label} ({len(qasper_ds)} examples)")
    print(f"{'='*60}")

    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    log_every = max(1, len(qasper_ds) // 10)

    for i, row in enumerate(qasper_ds):
        kwargs = dict(
            question="",
            answer_prefix="",
            max_new_tokens=MAX_NEW_TOKENS,
        )
        if press is not None:
            kwargs["press"] = press

        t_start = time.perf_counter()
        output = pipe(row["input"], **kwargs)
        elapsed = time.perf_counter() - t_start

        all_results.append({
            "framework": "kvpress",
            "press": press_name,
            "compression_ratio": ratio,
            "predicted_answer": output["answer"],
            "reference_answer": row["output"],
            "elapsed_sec": round(elapsed, 3),
        })

        if (i + 1) % log_every == 0 or (i + 1) == len(qasper_ds):
            total_elapsed = time.perf_counter() - t0
            print(f"  {i+1}/{len(qasper_ds)} — {total_elapsed:.0f}s elapsed")

    total_elapsed = time.perf_counter() - t0
    peak_mem = torch.cuda.max_memory_allocated() / 1e9
    print(f"  Done: {total_elapsed:.0f}s, peak_mem={peak_mem:.2f}GB")

    torch.cuda.empty_cache()

print(f"\nTotal results: {len(all_results)}")

## 5. Score & Results

Score predictions using the HuggingFace `evaluate` SQuAD metric
(token-level F1). SCROLLS references may be pipe-delimited for
multiple valid answers.

In [ ]:
import pandas as pd

df = pd.DataFrame(all_results)

all_metrics = {}
rows = []
for (press, ratio), group in df.groupby(["press", "compression_ratio"]):
    predictions = [
        {"id": str(i), "prediction_text": row["predicted_answer"]}
        for i, (_, row) in enumerate(group.iterrows())
    ]
    references = [
        {
            "id": str(i),
            "answers": {
                "text": row["reference_answer"].split("|"),
                "answer_start": [0] * len(row["reference_answer"].split("|")),
            },
        }
        for i, (_, row) in enumerate(group.iterrows())
    ]

    result = squad_metric.compute(predictions=predictions, references=references)
    key = f"{press}__{ratio}"
    all_metrics[key] = result
    mean_time = group["elapsed_sec"].mean()
    rows.append({
        "press": press, "compression_ratio": ratio,
        "f1": round(result["f1"], 2),
        "exact_match": round(result["exact_match"], 2),
        "mean_time": round(mean_time, 3),
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

## 6. Save Results

In [ ]:
import json

os.makedirs("results/kvpress_qasper", exist_ok=True)

predictions_path = "results/kvpress_qasper/predictions.csv"
df.to_csv(predictions_path, index=False)
print(f"Saved predictions to {predictions_path}")

metrics_path = "results/kvpress_qasper/metrics.json"
with open(metrics_path, "w") as f:
    json.dump(all_metrics, f, indent=2)
print(f"Saved metrics to {metrics_path}")